In [ ]:
!pip install transformers datasets librosa torchaudio accelerate matplotlib captum

In [ ]:
import torch
import numpy as np
from transformers import TrainingArguments, Trainer
import matplotlib.pyplot as plt
import seaborn as sns

###Φόρτωση συνόλου και προετοιμασία


In [ ]:
from datasets import load_dataset, Audio, DatasetDict
from sklearn.model_selection import GroupShuffleSplit

dataset = load_dataset("narad/ravdess", revision="refs/convert/parquet")["train"]
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

print("Πριν το split:", dataset.column_names)

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=dataset["speaker_id"]))

dataset = DatasetDict({
    "train": dataset.select(train_idx),
    "test": dataset.select(test_idx)
})

print("Μετά το split:", dataset["train"].column_names)
print(dataset["train"][0]["labels"])

In [ ]:
from collections import Counter

test = dataset["test"]
print("Δείγματα ελέγχου:", len(test))

for col in ("actor_id", "speaker_id", "speaker", "actor"):
    if col in test.column_names:
        spk = sorted(set(test[col]))
        print(f"{col}: {len(spk)} ομιλητές ->", spk)
        break
else:
    print("Δεν βρέθηκε πεδίο ομιλητή. Στήλες:", test.column_names)

label_col = "emotion" if "emotion" in test.column_names else test.column_names[-1]
print(Counter(test[label_col]))

In [ ]:
dataset_raw = load_dataset("narad/ravdess", revision="refs/convert/parquet")["train"]

label_names = dataset_raw.features["labels"].int2str(list(range(dataset_raw.features["labels"].num_classes)))
num_labels = dataset_raw.features["labels"].num_classes
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}
print(label_names)

In [ ]:
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
import torch

checkpoint = "facebook/hubert-base-ls960"
feature_extractor = AutoFeatureExtractor.from_pretrained(checkpoint)

model = AutoModelForAudioClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
).to("cuda")

model.freeze_feature_encoder()

nan_params = sum(torch.isnan(p).any().item() for p in model.parameters())
print(f"{nan_params} NaN parameters (πρέπει να είναι 0)")

In [ ]:
columns_to_remove = [c for c in dataset["train"].column_names if c != "labels"]
print(columns_to_remove)

def prepare_dataset(batch):
    audio = batch["audio"]
    max_samples = 6 * 16000
    audio_array = audio["array"][:max_samples]
    batch["input_features"] = feature_extractor(audio_array, sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = int(batch["labels"])
    return batch

encoded_dataset = dataset.map(prepare_dataset, remove_columns=columns_to_remove)

print(encoded_dataset["train"].column_names)
print(encoded_dataset["train"][0]["labels"], type(encoded_dataset["train"][0]["labels"]))

In [ ]:
from collections import Counter
print(Counter(dataset["train"]["labels"] if "labels" in dataset["train"].column_names else dataset["train"]["emotion"]))

### Ορισμός μετρικών αξιολόγησης

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,

    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,

    roc_auc_score,
    average_precision_score
)

from sklearn.preprocessing import label_binarize
from scipy.special import softmax

### Ορισμός παραμέτρων εκπαίδευσης

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from transformers import TrainingArguments, Trainer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, cohen_kappa_score, roc_auc_score, log_loss,
    confusion_matrix, average_precision_score
)
from sklearn.preprocessing import label_binarize
from scipy.special import softmax

class DataCollatorCTCWithPadding:
    def __init__(self, feature_extractor, padding=True):
        self.feature_extractor = feature_extractor
        self.padding = padding

    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        batch = self.feature_extractor.pad(input_features, padding=self.padding, return_tensors="pt")
        batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return batch

data_collator = DataCollatorCTCWithPadding(feature_extractor=feature_extractor, padding=True)

all_confusion_matrices = []

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=1)
    num_classes = probs.shape[1]
    labels_bin = label_binarize(labels, classes=list(range(num_classes)))

    try:
        roc_auc = roc_auc_score(labels_bin, probs, multi_class="ovr", average="macro")
    except ValueError:
        roc_auc = float("nan")
    try:
        pr_auc = average_precision_score(labels_bin, probs, average="macro")
    except ValueError:
        pr_auc = float("nan")

    accuracy = accuracy_score(labels, predictions)
    balanced_accuracy = balanced_accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average="weighted", zero_division=0)
    recall = recall_score(labels, predictions, average="weighted", zero_division=0)
    f1 = f1_score(labels, predictions, average="weighted", zero_division=0)
    mcc = matthews_corrcoef(labels, predictions)
    k_score = cohen_kappa_score(labels, predictions)
    logloss = log_loss(labels, probs, labels=list(range(num_classes)))

    cm = confusion_matrix(labels, predictions)
    all_confusion_matrices.append(cm)

    return {
        "accuracy": accuracy, "balanced_accuracy": balanced_accuracy,
        "precision": precision, "recall": recall, "f1": f1,
        "mcc": mcc, "k_score": k_score, "log_loss": logloss,
        "roc_auc": roc_auc, "pr_auc": pr_auc
    }



In [ ]:
# Δυναμικός υπολογισμός warmup_steps (10% των συνολικών training steps)
per_device_train_batch_size = 4
gradient_accumulation_steps = 4
num_train_epochs = 10

effective_batch_size = per_device_train_batch_size * gradient_accumulation_steps
steps_per_epoch = len(encoded_dataset["train"]) // effective_batch_size
total_steps = steps_per_epoch * num_train_epochs
warmup_steps = max(1, int(0.1 * total_steps))

training_args = TrainingArguments(
    output_dir="./hubert-finetuned-RAVDESS",
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=gradient_accumulation_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=num_train_epochs,
    fp16=False,
    max_grad_norm=1.0,
    logging_steps=10,
    logging_strategy="steps",
    learning_rate=2e-5,
    warmup_steps=warmup_steps,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

batch = next(iter(trainer.get_train_dataloader()))
batch = {k: v.to(model.device) for k, v in batch.items()}
with torch.no_grad():
    output = model(**batch)
print("Loss:", output.loss.item())
print("Any NaN:", torch.isnan(output.logits).any().item())

In [ ]:
trainer.train()

In [ ]:
eval_results = trainer.evaluate()
print(eval_results)

In [ ]:
#Αποθήκευση των ετικετών για τους άξονες των matrices
labels = [
    model.config.id2label[i]
    for i in sorted(model.config.id2label.keys())
]

#Loop στην λίστα με τα confusion matrices
for epoch, cm in enumerate(all_confusion_matrices):
  #Δημιοργία γραφήματος
  plt.figure(figsize=(6,5))

  #Δηνιουργία Heatmap
  sns.heatmap(
      cm,
      annot=True,
      fmt="d",
      xticklabels=labels,
      yticklabels=labels
  )

  plt.title(f"Confusion Matrix - Epoch {epoch+1}")
  plt.xlabel("Predicted")
  plt.ylabel("True")
  plt.show()

In [ ]:
repo_id = "VasilisAsim/hubert-finetuned-Ravdess"

print("Θα κάνω push στο:", repo_id)

print(f"Το μοντέλο και ο επεξεργαστής αποθηκεύτηκαν στο Hugging Face Hub: {repo_id}")

In [ ]:
model.push_to_hub(repo_id)
feature_extractor.push_to_hub(repo_id)

In [ ]:
from huggingface_hub import HfApi
info = HfApi().repo_info(repo_id, revision="main")
print("Νέο SHA στο main:", info.sha)